In [14]:
import torch
from torch import nn

In [15]:
class DoubleConvolution(nn.Module):
    """Spatial size를 유지하면서 feature channel을 변환"""
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        
        self.layers = nn.Sequential(
            
            # [B, 1, 64, 64] -> [B, 16, 64, 64]
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            # inplace=True: ReLU 결과를 새 tensor에 만들지 말고, 기존 tensor 값을 직접 덮어씀
            nn.ReLU(inplace=True),
            
            # [B, 16, 64, 64] -> [B, 16, 64, 64]
            nn.Conv2d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        
    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        return self.layers(input_tensor)
    

# Shape: [B, C, H, W]
encoder_input = torch.randn(
    2,
    1,
    64,
    64,
)

feature_extractor = DoubleConvolution(
    in_channels=1,
    out_channels=16,
)

encoder_features = feature_extractor(
    encoder_input,
)

print("Input shape: ", encoder_input.shape)
print("Output shape:", encoder_features.shape)

Input shape:  torch.Size([2, 1, 64, 64])
Output shape: torch.Size([2, 16, 64, 64])


In [16]:
class EncoderBlock(nn.Module):
    """Feature를 추출하고 spatial size를 절반으로 줄인다."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        
        # [2, 1, 64, 64] -> [2, 16, 64, 64]
        self.feature_extractor = DoubleConvolution(
            in_channels=in_channels,
            out_channels=out_channels,
        )
        
        # [2, 16, 64, 64] -> [2, 16, 32, 32]
        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )
        
    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        
        # decoder에 나중에 전달 -> 해상도 높은 세부정보 보존
        skip_features = self.feature_extractor(input_tensor)

        # encoder 다음 stage로 전달 -> 더 작은 해상도에서 더 깊은 feature 추출
        pooled_features = self.pool(skip_features)

        return skip_features, pooled_features
        
        
encoder_block = EncoderBlock(
    in_channels=1,
    out_channels=16,
)

skip_features, pooled_features = encoder_block(
    encoder_input,
)

print("Encoder input:  ", encoder_input.shape)
print("Skip features:  ", skip_features.shape)
print("Pooled features:", pooled_features.shape)

Encoder input:   torch.Size([2, 1, 64, 64])
Skip features:   torch.Size([2, 16, 64, 64])
Pooled features: torch.Size([2, 16, 32, 32])


In [17]:
# bottleneck [2, 32, 32, 32]
#      │ ConvTranspose2d
#      ▼
# upsampled  [2, 16, 64, 64]
#      │
#      ├── concatenate(dim=1) ◄── skip [2, 16, 64, 64]
#      ▼
# combined   [2, 32, 64, 64]
#      │ DoubleConvolution
#      ▼
# output     [2, 16, 64, 64]

class DecoderBlock(nn.Module):
    """Feature map을 확대하고 encoder의 skip feature와 결합"""
    
    def __init__(
        self,
        in_channels: int,
        skip_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        
        # Decoder의 저해상도 feature를 2배 확대:
        # (ConvTranspose2d): H, W를 2배로 키우고, channel은 32 → 16으로 바꿈
        self.upsample = nn.ConvTranspose2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=2,
            stride=2,
        )

        # Upsampling feature와 skip feature를 "channel 차원"으로 연결한다.
        self.feature_fusion = DoubleConvolution(
            in_channels=out_channels + skip_channels,
            out_channels=out_channels,
        )
        
    
    def forward(
        self,
        input_tensor: torch.Tensor,
        skip_features: torch.Tensor,
    ) -> torch.Tensor:
        
        # bottleneck_features
        # → upsampled_features
        # → concatenated_features
        # → output_features
        
        # [2, 32, 32, 32] -> [2, 16, 64, 64]
        upsampled_features = self.upsample(input_tensor)
        
        # [2, 16, 64, 64] + [2, 16, 64, 64] 
        # concatenated -> [2, 32, 64, 64]
        concatenated_features = torch.cat(
            (upsampled_features, skip_features),
            dim=1,
        )
        
        # [2, 32, 64, 64] -> [2, 16, 64, 64]
        output_features = self.feature_fusion(
            concatenated_features,
        )

        return output_features
    
    

# Encoder의 pooled feature를 더 깊은 feature로 변환:
# [2, 16, 32, 32] -> [2, 32, 32, 32]
bottleneck = DoubleConvolution(
    in_channels=16,
    out_channels=32,
)

bottleneck_features = bottleneck(
    pooled_features,
)

decoder_block = DecoderBlock(
    in_channels=32,
    skip_channels=16,
    out_channels=16,
)

decoder_features = decoder_block(
    input_tensor=bottleneck_features,
    skip_features=skip_features,
)

print("Bottleneck:    ", bottleneck_features.shape)
print("Skip features: ", skip_features.shape)
print("Decoder output:", decoder_features.shape)

Bottleneck:     torch.Size([2, 32, 32, 32])
Skip features:  torch.Size([2, 16, 64, 64])
Decoder output: torch.Size([2, 16, 64, 64])


In [18]:
# Input:
# [B,1,64,64]
#    │ DoubleConv 1→16
#    ▼
# [B,16,64,64] ─────── skip ───────┐
#    │ MaxPool                      │
#    ▼                              │
# [B,16,32,32]                      │
#    │ Bottleneck DoubleConv 16→32  │
#    ▼                              │
# [B,32,32,32]                      │
#    │ ConvTranspose 32→16, ×2      │
#    ▼                              │
# [B,16,64,64] ◄────────────────────┘
#    │ cat(dim=1)
#    ▼
# [B,32,64,64]
#    │ DoubleConv 32→16
#    ▼
# [B,16,64,64]
#    │ 1×1 Conv 16→K
#    ▼
# [B,K,64,64]
#    │ argmax(dim=1)
#    ▼
# [B,64,64]


class OneLevelUNet(nn.Module):
    """한 단계의 encoder, bottleneck, decoder를 연결한 작은 U-Net."""

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        base_channels: int,
    ) -> None:
        super().__init__()

        # [B, in_channels, H, W]
        # -> skip:   [B, base_channels, H, W]
        # -> pooled: [B, base_channels, H/2, W/2]
        self.encoder = EncoderBlock(
            in_channels=in_channels,
            out_channels=base_channels,
        )

        # [B, base_channels, H/2, W/2]
        # -> [B, base_channels*2, H/2, W/2]
        self.bottleneck = DoubleConvolution(
            in_channels=base_channels,
            out_channels=base_channels * 2,
        )

        # input: [B, base_channels*2, H/2, W/2] -> (upsampled)
        # skip:  [B, base_channels, H, W]
        # -> [B, base_channels, H, W]
        self.decoder = DecoderBlock(
            in_channels=base_channels * 2,
            skip_channels=base_channels,
            out_channels=base_channels,
        )

        # [B, base_channels, H, W]
        # -> [B, num_classes, H, W]
        self.segmentation_head = nn.Conv2d(
            in_channels=base_channels,
            out_channels=num_classes,
            kernel_size=1,
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:

        # [B, in_channels, H, W]
        # -> skip_features:   [B, base_channels, H, W]
        # -> pooled_features: [B, base_channels, H/2, W/2]
        skip_features, pooled_features = self.encoder(
            input_tensor,
        )

        # [B, base_channels, H/2, W/2]
        # -> [B, base_channels*2, H/2, W/2]
        bottleneck_features = self.bottleneck(
            pooled_features,
        )

        # bottleneck: [B, base_channels*2, H/2, W/2]
        # skip:       [B, base_channels, H, W]
        # ->          [B, base_channels, H, W]
        decoder_features = self.decoder(
            input_tensor=bottleneck_features,
            skip_features=skip_features,
        )

        # [B, base_channels, H, W]
        # -> [B, num_classes, H, W]
        segmentation_logits = self.segmentation_head(
            decoder_features,
        )

        return segmentation_logits


model = OneLevelUNet(
    in_channels=1,
    num_classes=3,
    base_channels=16,
)

input_images = torch.randn(
    2,
    1,
    64,
    64,
)

segmentation_logits = model(
    input_images,
)

predictions = segmentation_logits.argmax(
    dim=1,
)

print("Input:      ", input_images.shape)
print("Logits:     ", segmentation_logits.shape)
print("Prediction: ", predictions.shape)

Input:       torch.Size([2, 1, 64, 64])
Logits:      torch.Size([2, 3, 64, 64])
Prediction:  torch.Size([2, 64, 64])
